# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and interact with the FAIR² dataset ["Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells"](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the [`mlcroissant`](https://github.com/mlcommons/croissant) Python library.

We explore its Croissant-based schema, extract record sets and columns with field `@id` references, and perform analysis.

### Dataset Source
The dataset Croissant schema is published at:
[https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Display dataset metadata
print(f"Title: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Published: {dataset.metadata.date_published}")
print(f"Version: {dataset.metadata.version}")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview
Review available record sets and their fields. All entities are referenced by their Croissant schema `@id`s.

Below we enumerate the available record sets, fields, and corresponding column IDs defined in the Croissant schema.

In [ ]:
# List all Record Sets by their @id
print("Available Record Sets (by @id):")
for record_set in dataset.metadata.record_sets:
    rec_id = record_set.id
    print(f"  - @id: {rec_id}, name: {record_set.name}")
    print("    Fields:")
    for field in record_set.fields:
        field_id = field.id
        column_ids = [c.id for c in field.columns]
        print(f"      - @id: {field_id}, name: {field.name}, columns: {[x for x in column_ids]}")

## 3. Data Extraction
Let's extract all tabular data from the main record set. We use `@id` of the desired record set and load its records into a DataFrame.

> In this dataset, the main record set typically contains protein abundance and description data. Consult the previous cell's output to select the primary record set by its `@id`.

In [ ]:
# Gather all record set IDs from the Croissant metadata
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
print("Record set @ids available:", record_set_ids)

# For this dataset, there is typically one main record set. We'll select the first one.
primary_record_set_id = record_set_ids[0] if len(record_set_ids) > 0 else None
if not primary_record_set_id:
    print("No record sets found in the metadata.")
else:
    # Read records for the primary record set into a DataFrame
    records = list(dataset.records(record_set=primary_record_set_id))
    df = pd.DataFrame(records)
    print(f"Loaded DataFrame columns from record set '@id={primary_record_set_id}':")
    print(df.columns.tolist())
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps (filtering, normalization, grouping) on numeric fields. All fields are referenced by their Croissant schema `@id`s. 

For this dataset, we'll pick a typical numeric field such as peptide count or molecular weight. Use the output of the previous overview/code cell to select an actual field `@id` for analysis.

In [ ]:
# Example: Choose a numeric field such as molecular weight (MW) or "Peptide_Count"
# Replace the string below with the actual @id of a numeric field as shown in previous cells' output.
numeric_field_id = None
# Try to automatically pick a numeric column name that commonly appears
candidate_numeric_fields = [col for col in df.columns if any(x in col.lower() for x in ["molecular", "mw", "count", "abundance", "coverage"])]
if candidate_numeric_fields:
    numeric_field_id = candidate_numeric_fields[0]
    print(f"Selected numeric field: {numeric_field_id}")
else:
    print("No commonly-named numeric field found. Please select from:", df.columns.tolist())

if numeric_field_id is not None:
    # Ensure the field is numeric
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].quantile(0.9)  # Use 90th percentile for demonstration
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (90th percentile):")
    print(filtered_df.head())

    # Normalize the selected numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by a likely categorical/label/text field (e.g., protein description or accession)
    candidate_group_fields = [col for col in df.columns if any(x in col.lower() for x in ["accession", "description", "sample"])]
    group_field = candidate_group_fields[0] if candidate_group_fields else None

    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field}, mean {numeric_field_id} for high-abundance records:")
        print(grouped_df.head())
    else:
        print("No group field found for grouping operation.")
else:
    print("No numeric field selected for EDA.")

## 5. Visualization
Visualize distributions or field relationships for the selected numeric and group fields.

Here we plot the filtered and normalized molecular weight or abundance for the top proteins, grouped by their IDs (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 5))
if numeric_field_id is not None and group_field:
    # Bar plot of group vs. mean normalized value (only top 20 by default)
    top_groups = grouped_df.sort_values(numeric_field_id, ascending=False).head(20)
    sns.barplot(x=group_field, y=numeric_field_id, data=top_groups)
    plt.xticks(rotation=90)
    plt.title(f"Top 20 {group_field} by {numeric_field_id}")
    plt.tight_layout()
    plt.show()
elif numeric_field_id is not None:
    sns.histplot(df[numeric_field_id].dropna(), bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.show()
else:
    print("No suitable numeric field or group field for visualization.")

## 6. Conclusion

- We loaded and explored the FAIR² dataset using a Croissant schema and the `mlcroissant` library.
- We accessed record sets, fields, and columns using their `@id` identifiers as per the schema specification.
- We performed filtering, normalization, grouping, and visualization of protein data (such as molecular weight or peptide abundance).

This approach enables robust, schema-driven data exploration for complex FAIR datasets.